# Notebook → Module Refactor Kit

Companion for **Moving from notebooks to real team code**. Self-contained — it writes its own tiny
sample file, so it runs anywhere with no setup. Goal: turn one throwaway notebook cell into
reusable, tracked, team-ready code.

## BEFORE — the cell your manager just saw

```python
import pandas as pd

df = pd.read_csv('/Users/rahul/Downloads/sales_data_final_v3.csv')
df = df.dropna(subset=['revenue'])
df['revenue'] = df['revenue'].astype(float)
df = df[df['revenue'] > 0]
df['month'] = pd.to_datetime(df['date']).dt.month
print(df.shape)
```

**Three things that break for a teammate (Task 1):** hardcoded laptop-only path · no function, so
nobody can reuse it · prints a side effect instead of returning data · the `> 0` threshold is
baked in. (Not run here — that path only exists on Rahul's machine.)

In [ ]:
# Setup — write a tiny messy sample so the refactored function has something to read.
import pandas as pd
from pathlib import Path

Path("sample_sales.csv").write_text(
    "date,revenue,region\n"
    "2024-01-05,1200,North\n"
    "2024-01-06,,South\n"      # missing revenue
    "2024-01-07,-50,East\n"    # invalid revenue
    "2024-01-08,900,West\n"
    "2024-02-01,1500,North\n",
    encoding="utf-8",
)
print("wrote sample_sales.csv")

In [ ]:
# AFTER — Tasks 2 + 4: a parameterised function with a docstring and type hints.
def clean_sales_data(path: str, min_revenue: float = 0) -> pd.DataFrame:
    """Load raw sales data and return a cleaned frame.

    Args:
        path: path to the sales CSV.
        min_revenue: drop rows at or below this revenue (default 0).

    Returns:
        A cleaned DataFrame with a numeric `revenue` and an added `month` column.
    """
    df = pd.read_csv(path)
    df = df.dropna(subset=["revenue"])
    df["revenue"] = df["revenue"].astype(float)
    df = df[df["revenue"] > min_revenue]
    df["month"] = pd.to_datetime(df["date"]).dt.month
    return df


clean_sales_data("sample_sales.csv")

In [ ]:
# A tiny test — no pytest needed. Run it; if nothing prints an error, you are good.
out = clean_sales_data("sample_sales.csv")
assert "month" in out.columns
assert (out["revenue"] > 0).all()           # no zero / negative revenue survived
assert out["revenue"].dtype == float
print("All checks passed:", len(out), "clean rows")

## Task 3 + 5 — move it to a module, then track it

```
my_project/
├── notebooks/
│   └── exploration.ipynb
├── src/
│   └── data_utils.py      ← clean_sales_data lives HERE, not in the notebook
└── main.py
```

Import it from `main.py`:

```python
from src.data_utils import clean_sales_data
```

Version it (three commands, in order):

```bash
git status
git add src/data_utils.py
git commit -m "Add clean_sales_data: reusable sales cleaning for the team"
```

**Do this now:** paste your function into `src/data_utils.py`, import it once, and make that commit.
A teammate should be able to run it on a new file without asking you a single question.